# Lab 03 · Pandas for Data Manipulation and Cleaning
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais

**CLO 6** (C-4 Analyze, PLO 2 Problem Analysis): *Analyze real-world datasets by applying data manipulation and visualization techniques.*

**Data:** `pakistan_used_cars.csv`, 24,973 real used-car ads from OLX Pakistan (Open Data Pakistan, *Pakistan Used Cars*, 2021). The ads are real and contain the problems real data has. Two kinds of formatting problem were added for practice (some prices written as text such as `Rs 1,430,000`, some city names in the wrong case or with extra spaces).

**Rules:** no Python loops over rows · label every output, e.g. `print("T4e:", ...)` · do not edit the CSV by hand · Runtime → Restart and run all before you submit · push this notebook to `ML-Lab-<RollNo>/Lab03/Lab03.ipynb` by **Thursday 5 November 2026, 11:59 PM (PKT)**.

Name: ______  Roll No: ______

## Warm-up (not graded): the class example from the lecture
Run this cell and repeat any lecture example on this small table before you start the tasks. `clean` is the tidy version, `messy` has the six kinds of problem.

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.width', 140)

clean = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Zainab"],
    "City": ["Muzaffarabad", "Mirpur", "Bagh", "Muzaffarabad", "Rawalakot", "Mirpur", "Kotli", "Muzaffarabad"],
    "Maths": [78, 91, 45, 85, 62, 73, 38, 88],
    "Physics": [65, 88, 52, 79, 48, 81, 41, 92],
    "Attendance": [92, 97, 70, 88, 75, 90, 60, 95]})

messy = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-04", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Fatima", "Zainab"],
    "City": [" Muzaffarabad", "Mirpur", "Bagh", "MUZAFFARABAD", "Rawalakot", "Mirpur", "Kotli", "MUZAFFARABAD", "Muzafarabad"],
    "Maths": [78, 91, np.nan, 85, 62, 73, 380, 85, 88],
    "Physics": [65, 88, 52, 79, 48, 105, 41, 79, 92],
    "Attendance": ["92%", "97", "70", "88", "75%", "90", "60", "88", "95"]})
print(clean)

    Roll    Name          City  Maths  Physics  Attendance
0  SE-01     Ali  Muzaffarabad     78       65          92
1  SE-02  Ayesha        Mirpur     91       88          97
2  SE-03   Bilal          Bagh     45       52          70
3  SE-04  Fatima  Muzaffarabad     85       79          88
4  SE-05   Hamza     Rawalakot     62       48          75
5  SE-06    Sana        Mirpur     73       81          90
6  SE-07   Usman         Kotli     38       41          60
7  SE-08  Zainab  Muzaffarabad     88       92          95


In [2]:
# Try lecture examples here, e.g. 
clean[clean['Maths'] > 80]


,Roll,Name,City,Maths,Physics,Attendance
1,SE-02,Ayesha,Mirpur,91,88,97
3,SE-04,Fatima,Muzaffarabad,85,79,88
7,SE-08,Zainab,Muzaffarabad,88,92,95


### Setup
Put `pakistan_used_cars.csv` in the same folder as this notebook. In Colab: click the folder icon on the left → Upload → choose the CSV.

## Task 1 · Load and first look
- **(a)** Read `pakistan_used_cars.csv` into `df` and print its shape.
- **(b)** Show the first 5 rows.
- **(c)** Print the column names.
- **(d)** Show the data type of every column.
- **(e)** Which column should be a number but is not? Show two of its values that explain why.

In [3]:

# Task 1: Load and first look

# (a) Read the CSV file and print its shape
df = pd.read_csv("pakistan_used_cars.csv")
print("T1a - Dataset Shape:", df.shape)

# (b) Display the first 5 rows
print("\nT1b - First 5 Rows:")
print(df.head(5))

# (c) Display all column names
print("\nT1c - Column Names:")
print(df.columns.tolist())

# (d) Display data types of all columns
print("\nT1d - Data Types:")
print(df.dtypes)

# (e) Show two price values stored as text
price_examples = df.loc[
    df["Price"].str.contains(r"Rs|,", na=False), "Price"
].head(2)

print("\nT1e - Price Values Stored as Text:")
print(price_examples.tolist())



T1a - Dataset Shape: (24973, 9)

T1b - First 5 Rows:
    Brand Condition    Fuel  KMs Driven        Model    Price Registered City Transaction Type    Year
0  Toyota      Used  Diesel         1.0        Prado  2100000         Karachi             Cash  1997.0
1  Suzuki      Used  Petrol    100000.0        Bolan   380000         Karachi             Cash  2006.0
2  Suzuki      Used     CNG     12345.0        Bolan   340000         Karachi             Cash  1998.0
3  Suzuki      Used  Petrol     94000.0         Alto   535000         Karachi             Cash  2010.0
4  Toyota      Used  Petrol    100000.0  Corolla XLI  1430000         Karachi             Cash  2013.0

T1c - Column Names:
['Brand', 'Condition', 'Fuel', 'KMs Driven', 'Model', 'Price', 'Registered City', 'Transaction Type', 'Year']

T1d - Data Types:
Brand                   str
Condition               str
Fuel                    str
KMs Driven          float64
Model                   str
Price                   str
Registered 

**T1e – Explanation:**

The Price column should contain numerical values because it represents the prices of cars. However, it is stored as an object (text) because some prices contain currency symbols such as "Rs" and commas, for example, "Rs 435,000" and "Rs 4,150,000". These characters prevent Pandas from treating the values as numbers. Therefore, the formatting characters need to be removed before converting the Price column into a numerical data type.

## Task 2 · Retrieve data
- **(a)** The `Brand` column (first 5 values).
- **(b)** The columns `Brand`, `Model`, `Price` together (first 5 rows).
- **(c)** The first row, using `iloc`.
- **(d)** The `Model` in the row with index 100, using `loc`.
- **(e)** How many cars are registered in `"Islamabad"`?
- **(f)** How many **Honda** cars run on **Hybrid** fuel?
- **(g)** Cars from 2018 or newer, newest first (Brand, Model, Year, Price; first 10).
- **(h)** Try `df[df["Price"] > 1_000_000]`. What happens and why? Answer in a Markdown cell.

In [4]:

# Task 2: Data Retrieval and Filtering

# (a) Display one column: Brand
print("T2a - Brand Column:")
print(df["Brand"])

# (b) Display three columns: Brand, Model, and Price
print("\nT2b - Brand, Model, and Price:")
print(df[["Brand", "Model", "Price"]])

# (c) Display the first row using iloc
print("\nT2c - First Row:")
print(df.iloc[0])

# (d) Display the Model value at row index 100 using loc
print("\nT2d - Model at Row Index 100:")
print(df.loc[100, "Model"])

# (e) Count cars registered in Islamabad
islamabad_count = df[df["Registered City"] == "Islamabad"].shape[0]
print("\nT2e - Cars Registered in Islamabad:", islamabad_count)

# (f) Count Honda cars with Hybrid fuel
honda_hybrid_count = df[
    (df["Brand"] == "Honda") & (df["Fuel"] == "Hybrid")
].shape[0]
print("\nT2f - Honda Hybrid Cars:", honda_hybrid_count)

# (g) Display the first 10 cars from 2018 onwards,
# sorted by newest year first
recent_cars = df[df["Year"] >= 2018][
    ["Brand", "Model", "Year", "Price"]
].sort_values(by="Year", ascending=False).head(10)

print("\nT2g - First 10 Cars from 2018 Onwards:")
print(recent_cars)

# (h) Try filtering cars with a price above 1,000,000
print("\nT2h - Testing Price Filtering:")

try:
    expensive_cars = df[df["Price"] > 1_000_000]
    print(expensive_cars.head())
except TypeError as error:
    print("Comparison failed:", error)



T2a - Brand Column:
0              Toyota
1              Suzuki
2              Suzuki
3              Suzuki
4              Toyota
             ...     
24968          Toyota
24969        Daihatsu
24970    Other Brands
24971          Suzuki
24972          Toyota
Name: Brand, Length: 24973, dtype: str

T2b - Brand, Model, and Price:
              Brand        Model    Price
0            Toyota        Prado  2100000
1            Suzuki        Bolan   380000
2            Suzuki        Bolan   340000
3            Suzuki         Alto   535000
4            Toyota  Corolla XLI  1430000
...             ...          ...      ...
24968        Toyota   Corolla XE  1070000
24969      Daihatsu        Cuore   390000
24970  Other Brands        Other   180000
24971        Suzuki         Alto   470000
24972        Toyota  Corolla GLI  2050000

[24973 rows x 3 columns]

T2c - First Row:
Brand                Toyota
Condition              Used
Fuel                 Diesel
KMs Driven              1.0
Model  

**T2h – Explanation:**

The price filtering operation fails because the Price column is stored as an object (text) data type instead of a numerical data type. Python cannot directly compare text values with an integer value such as 1,000,000. To perform this comparison correctly, the Price column must first be cleaned by removing currency symbols and commas and then converted into a numerical data type. This issue will be fixed during Task 3.

## Task 3 · Clean the data: the six types of problem
Fix the problems **in this order** (text first, then duplicates):
- **(a) Missing values:** count them per column. Drop ads with no `Brand` or no `Model`; fill missing `Registered City`, `Fuel` and `Transaction Type` with `"Unknown"`; leave `Year` and `KMs Driven` as NaN. Print how many rows were removed.
- **(b) Formatting:** remove extra spaces and fix the case of city names (`.str.strip().str.title()`); remove `"Rs"` and commas from `Price`.
- **(c) Data types:** convert `Price` to `int64` and `Year` to `Int64`.
- **(d) Inconsistent categories:** `"Sukkar"` → `"Sukkur"` and `"Corrolla Altis"` → `"Corolla Altis"`. Print the number of different cities before and after.
- **(e) Duplicates:** count exact duplicate ads, drop them, print the new row count.
- **(f) Invalid values:** set `KMs Driven` to NaN when it is below 100, above 1,000,000, or the placeholder 12345 / 123456. Drop ads with the fake prices 12,345,678 or 87,654,321, and Suzuki/Daihatsu ads above 5,000,000.
- **(g) Self-check:** your final shape must be **(20484, 9)**.

In [5]:

# Task 3: Data Cleaning

# Start again with the original dataset
df = pd.read_csv("pakistan_used_cars.csv")

# (a) Handle missing values
print("T3a - Missing Values Before Cleaning:")
print(df.isnull().sum())

# Remove advertisements with missing Brand or Model
rows_before = len(df)
df = df.dropna(subset=["Brand", "Model"])
rows_removed = rows_before - len(df)

print("T3a - Rows Removed:", rows_removed)

# Fill selected missing values with Unknown
df["Registered City"] = df["Registered City"].fillna("Unknown")
df["Fuel"] = df["Fuel"].fillna("Unknown")
df["Transaction Type"] = df["Transaction Type"].fillna("Unknown")

# (b) Clean formatting
df["Registered City"] = (
    df["Registered City"].str.strip().str.title()
)

df["Price"] = (
    df["Price"]
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

print("\nT3b - City and Price Formatting Completed.")

# (c) Convert data types
df["Price"] = pd.to_numeric(df["Price"]).astype("int64")
df["Year"] = df["Year"].astype("Int64")

print("\nT3c - Updated Data Types:")
print(df[["Price", "Year"]].dtypes)

# (d) Correct inconsistent categories
cities_before = df["Registered City"].nunique()

df["Registered City"] = df["Registered City"].replace({
    "Sukkar": "Sukkur"
})

df["Model"] = df["Model"].replace({
    "Corrolla Altis": "Corolla Altis"
})

cities_after = df["Registered City"].nunique()

print("\nT3d - Different Cities Before Correction:", cities_before)
print("T3d - Different Cities After Correction:", cities_after)

# (e) Remove exact duplicate advertisements
duplicates_count = df.duplicated().sum()
df = df.drop_duplicates()

print("\nT3e - Duplicate Advertisements Removed:", duplicates_count)
print("T3e - Rows After Removing Duplicates:", len(df))

# (f) Handle invalid values
# Set unrealistic or placeholder kilometre values to missing
invalid_km = (
    (df["KMs Driven"] < 100) |
    (df["KMs Driven"] > 1000000) |
    (df["KMs Driven"].isin([12345, 123456]))
)

df.loc[invalid_km, "KMs Driven"] = np.nan

# Remove advertisements with fake prices
fake_price = df["Price"].isin([12345678, 87654321])

# Remove Suzuki and Daihatsu advertisements above 5 million
invalid_brand_price = (
    df["Brand"].isin(["Suzuki", "Daihatsu"]) &
    (df["Price"] > 5000000)
)

df = df[~(fake_price | invalid_brand_price)].copy()

print("\nT3f - Invalid Values Handled.")

# Final self-check
print("\nT3g - Final Dataset Shape:", df.shape)



T3a - Missing Values Before Cleaning:
Brand               2137
Condition           2136
Fuel                2445
KMs Driven          2286
Model               2448
Price                  0
Registered City     4636
Transaction Type    2445
Year                2284
dtype: int64
T3a - Rows Removed: 2448

T3b - City and Price Formatting Completed.

T3c - Updated Data Types:
Price    int64
Year     Int64
dtype: object

T3d - Different Cities Before Correction: 62
T3d - Different Cities After Correction: 61

T3e - Duplicate Advertisements Removed: 2028
T3e - Rows After Removing Duplicates: 20497

T3f - Invalid Values Handled.

T3g - Final Dataset Shape: (20484, 9)


## Task 4 · Analyse
- **(a)** New column `Price_lakh` = price in lakh rupees (÷ 100,000).
- **(b)** New column `Age` = 2020 − `Year`.
- **(c)** The 5 most common brands.
- **(d)** The 5 cities with the most ads.
- **(e)** Median `Price_lakh` for each fuel type, highest first.
- **(f)** Median `Price_lakh` of a **Corolla GLI** for each year 2010–2018. How much value does it lose per year?
- **(g)** The 5 most expensive ads. Does one of them look wrong? Why?
- **(h)** Number of ads and median price for Cash vs Installment/Leasing. Interpret in 1–2 lines.

In [6]:

# Task 4: Data Analysis

# (a) Convert car prices into lakhs
df["Price_lakh"] = df["Price"] / 100000
print("T4a - Price in Lakhs:")
print(df[["Price", "Price_lakh"]].head())

# (b) Calculate the age of each car
df["Age"] = 2020 - df["Year"]
print("\nT4b - Car Age:")
print(df[["Year", "Age"]].head())

# (c) Find the 5 most common car brands
print("\nT4c - Top 5 Car Brands:")
print(df["Brand"].value_counts().head(5))

# (d) Find the 5 cities with the most advertisements
print("\nT4d - Top 5 Cities:")
print(df["Registered City"].value_counts().head(5))

# (e) Find the median price for each fuel type
fuel_median = df.groupby("Fuel")["Price_lakh"].median().sort_values(
    ascending=False
)
print("\nT4e - Median Price by Fuel Type:")
print(fuel_median)

# (f) Find median prices of Toyota Corolla GLI cars from 2010 to 2018
corolla_gli = df[
    (df["Brand"] == "Toyota") &
    (df["Model"].str.contains("Corolla GLI", case=False, na=False)) &
    (df["Year"].between(2010, 2018))
]

corolla_gli_by_year = corolla_gli.groupby("Year")["Price_lakh"].median()

print("\nT4f - Corolla GLI Median Price by Year:")
print(corolla_gli_by_year)

# Calculate average yearly change in median price
if len(corolla_gli_by_year) >= 2:
    first_year = corolla_gli_by_year.index.min()
    last_year = corolla_gli_by_year.index.max()

    yearly_loss = (
        corolla_gli_by_year.loc[first_year] -
        corolla_gli_by_year.loc[last_year]
    ) / (last_year - first_year)

    print("T4f - Average Yearly Value Loss (Lakhs):", yearly_loss)
else:
    print("T4f - Not enough yearly data to calculate value loss.")

# (g) Find the 5 most expensive advertisements
most_expensive = df.nlargest(5, "Price_lakh")[
    ["Brand", "Model", "Year", "Price", "Price_lakh"]
]

print("\nT4g - Five Most Expensive Advertisements:")
print(most_expensive)

# (h) Compare Cash and Installment/Leasing advertisements
transaction_summary = (
    df[df["Transaction Type"].isin(["Cash", "Installment/Leasing"])]
    .groupby("Transaction Type")["Price_lakh"]
    .agg(["count", "median"])
)

print("\nT4h - Cash vs Installment/Leasing:")
print(transaction_summary)


T4a - Price in Lakhs:
     Price  Price_lakh
0  2100000       21.00
1   380000        3.80
2   340000        3.40
3   535000        5.35
4  1430000       14.30

T4b - Car Age:
   Year  Age
0  1997   23
1  2006   14
2  1998   22
3  2010   10
4  2013    7

T4c - Top 5 Car Brands:
Brand
Suzuki      8621
Toyota      4535
Honda       3025
Daihatsu    2231
Nissan       694
Name: count, dtype: int64

T4d - Top 5 Cities:
Registered City
Karachi       15461
Unknown        2040
Lahore         1601
Islamabad       777
Rawalpindi      119
Name: count, dtype: int64

T4e - Median Price by Fuel Type:
Fuel
Hybrid    16.2750
Diesel    13.0000
Petrol     8.7525
CNG        3.8500
LPG        1.5750
Name: Price_lakh, dtype: float64

T4f - Corolla GLI Median Price by Year:
Year
2010    12.250
2011    13.325
2012    14.125
2013    14.950
2014    16.750
2015    17.900
2016    19.000
2017    21.000
2018    23.200
Name: Price_lakh, dtype: float64
T4f - Average Yearly Value Loss (Lakhs): -1.36875

T4g - Five Mos

**T4f – Interpretation:**

The median prices of Toyota Corolla GLI cars show an increasing trend from 2010 to 2018. The average yearly value loss is calculated as -1.36875 lakhs, which actually indicates an average yearly price increase of approximately 1.37 lakhs. This shows that newer model-year Corolla GLI cars have higher median advertisement prices in this dataset.

**T4g – Interpretation:**

The five most expensive advertisements include Toyota Land Cruiser, Toyota Surf and Toyota Prius vehicles. The 2005 Toyota Land Cruiser listed at 520 lakhs looks unusual because its asking price is higher than the prices of the newer 2015 and 2017 Land Cruisers. This advertisement may require further investigation, although the available data alone cannot confirm that its price is incorrect.

**T4h – Interpretation:**

The dataset contains 19,634 Cash advertisements and 850 Installment/Leasing advertisements. The median price for Cash transactions is 6.9 lakhs, while the median price for Installment/Leasing transactions is 3.0 lakhs. This indicates that Cash advertisements have a higher median asking price than Installment/Leasing advertisements in this dataset.

## Task 5 · Save your work
Save the cleaned DataFrame as `pakistan_used_cars_clean.csv` (Lab 04 uses it) and print its shape.

In [7]:
# Task 5: Save the Cleaned Dataset

df.to_csv("pakistan_used_cars_clean.csv", index=False)

print("T5 - Cleaned dataset saved successfully.")
print("T5 - Final Dataset Shape:", df.shape)


T5 - Cleaned dataset saved successfully.
T5 - Final Dataset Shape: (20484, 11)


## Task 6 · Reflection (Markdown cell)
Write 3–5 sentences: which of the six cleaning problems would have given the most misleading answer if you had ignored it, and two things you learned about the used-car market in Pakistan.

The most important cleaning problem was handling incorrect and unrealistic values because they could have produced misleading results during data analysis. I learned that Toyota is an important brand in Pakistan's used-car market and that car prices vary according to the model year, fuel type, and transaction type. I also learned that Cash advertisements have a higher median price than Installment/Leasing advertisements in this dataset. This lab helped me understand how data cleaning improves the accuracy and reliability of analysis.